# Warm-up — what does a Python loop cost?

*~8 minutes. Fill **two** functions, run everything, read the numbers.*

You'll sum 10⁶ numbers three ways: your own `for` loop, `numpy`, and a `numba`-compiled loop. Same maths — very different speed. This is the whole question of module 1, in two measurements.

In [ ]:
import timeit, numpy as np

v = np.random.rand(1_000_000)     # a million numbers
v_list = v.tolist()               # the SAME data, as a plain Python list

def ms(t): return f"{t*1e3:.2f} ms"
def best(f, r=5):                 # time per call, %timeit-style
    n = 1
    while timeit.timeit(f, number=n) < 0.05: n *= 10
    return min(timeit.repeat(f, number=n, repeat=r)) / n

## 1. Your turn — the pure Python loop

Write `mysum`: return the sum of the elements of `x` with a plain `for` loop (no `sum`, no `numpy` — we *want* the slow version here).

In [ ]:
def mysum(x):
    ## SOLUTION: add up the elements of x with a for loop ##
    s = 0.0
    for xi in x:
        s += xi
    return s
    ## END ##

In [ ]:
# Run me — no need to edit.
assert abs(mysum(v_list) - sum(v_list)) < 1e-6, 'mysum should return the sum'
t_loop = best(lambda: mysum(v_list))
t_np   = best(lambda: np.sum(v))
print("your Python loop :", ms(t_loop))
print("numpy.sum        :", ms(t_np))
print(f"\n→ numpy is ~{t_loop/t_np:.0f}× faster — same maths, but it never enters the Python interpreter.")

## 2. Your turn — the *same* loop, compiled

`numba` compiles a Python function to native code — Julia's trick, bolted onto Python. Write the **same** loop again inside `sum_numba`; `@njit` will compile it.

In [ ]:
try:
    from numba import njit
except ImportError:                       # Colab already has numba
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'])
    from numba import njit

In [ ]:
@njit
def sum_numba(x):
    ## SOLUTION: the SAME loop as mysum — numba compiles it to native code ##
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s
    ## END ##

In [ ]:
# Run me — the 1st call compiles (once), then it's fast.
assert abs(sum_numba(v) - np.sum(v)) < 1e-6, 'sum_numba should return the sum'
t_nb = best(lambda: sum_numba(v))
print("your pure loop  :", ms(t_loop))
print("numba njit loop :", ms(t_nb))
print(f"\n→ ~{t_loop/t_nb:.0f}× faster than your pure loop — the SAME code, now compiled instead of interpreted.")

### What you just saw

Your loop pays the **interpreter** on every one of the million iterations. `numpy` sidesteps it (compiled C); `numba` **removes** it (it compiles *your* loop). "JIT" isn't magic — it's *specialize on the types, then compile*. That gap **is** module 1.

> The **Julia** twin writes the same hand loop with **no library at all** — because Julia is compiled by default. Open `1_warmup_julia.jl` and compare the numbers.